In [1]:
from dtgraph import Neo4jGraph, Rule, Transformation

hostname = "localhost"
password = "internship"
uri = f"bolt://{hostname}:7687"

graph = Neo4jGraph(uri, database="neo4j", username="neo4j", password=password)

In [2]:
import sys
import os

# Go to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add dtgraph folder to path
sys.path.append(os.path.join(project_root, "../dtgraph"))

In [3]:
from dtgraph.scenarios.movies import Movies

Movies.load(graph)

Flushed database: Deleted 312 nodes, deleted 355 relationships, completed after 8014 ms.
Load scenario: Added 171 labels, created 171 nodes, set 564 properties, created 253 relationships, completed after 12375 ms.


In [4]:
from type_checking.environment import Environment

env = Environment("../../dtgraph/type_checking/ENVs/env_movies.json")

##### Rules

In [5]:
generate_films = Rule(
    """
MATCH (m:Movie)
WHERE m.title IS NOT NULL
GENERATE
(x = (m):Film {
    name = m.title
})
""",
    env=env,
    type_strict=True,
)

Rule_TEST2 = Rule(
"""
MATCH (x:Person)-[:ACTED_IN]->(y:Movie)
GENERATE
(u = (x):Person {
    name = x.name
})-[():IS_AN {
    years = [y.released],
    movies = [y.title]
}]->(z = ():Actor)
""",env=env,type_strict=True,
)


In [6]:
from dtgraph.type_checking.check_types import check_types

check_types([generate_films], env)


--- Checking Rule ---
{'lhs': 'MATCH (m:Movie)\nWHERE m.title IS NOT NULL', 'constructors': [{'alias': 'x', 'ids': ['m'], 'labels': ['Film'], 'properties': [{'key': 'name', 'value': 'm.title\n'}]}]}
AST:
PropertyAccess
    ├── var: m
    └── prop: title

 All rules passed type checking



##### Applying Rules

In [6]:
my_transform = Transformation([generate_films, Rule_TEST2])
my_transform.apply_on(graph)

Index: Added 0 index, completed after 385 ms.
Before compiling LHS:  MATCH (m:Movie)
WHERE m.title IS NOT NULL
Updated LHS:
 MATCH (m:Movie)
WHERE m.title IS NOT NULL
Rule: Added 76 labels, created 38 nodes, set 76 properties, created 0 relationships, completed after 2333 ms.
Before compiling LHS:  MATCH (x:Person)-[:ACTED_IN]->(y:Movie)
Updated LHS:
 MATCH (x:Person)-[:ACTED_IN]->(y:Movie)
WHERE x.name IS NOT NULL AND y.released IS NOT NULL AND y.title IS NOT NULL
Rule: Added 206 labels, created 103 nodes, set 721 properties, created 102 relationships, completed after 2870 ms.


5203

##### Abort Transformation

In [ ]:
my_transform.abort()